# CS453 - Two-Week Individual Project Workbook

**Name:** [Amna ALshaaf]  
**Student ID:** [09191049]  
**Project:** [Campus Delivery]

Read the two-week student guide first. This workbook supplies only map data and a variant helper, not search or agent solutions. Complete your own explanations and implementations. Save this notebook in your private GitHub repository as project.ipynb and follow docs/github-colab-guide.md. Submit the repository and final commit links in Classroom. The full course project continues after the professor returns.


## 1. Task, PEAS and environment

Explain your chosen task, inputs, output and success criteria. Complete PEAS and justify observability, determinism, sequential/episodic, static/dynamic, discrete/continuous and number of agents for this initial simulation. What are three scope limits?


## 2. Representation

Define initial state, legal actions, result/transition, goal test and costs. Include task status as well as location. Define service legality and repeated-service behavior. State whether the requested task is initially complete.


In [35]:
# Supplied data/helper: cite CS453 two-week project workbook if adapted.
from copy import deepcopy
import platform

REFERENCE_GRAPH = {
    "S": ["A", "B"],
    "A": ["S", "C", "D"],
    "B": ["S", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "G"],
    "F": ["C", "G"],
    "G": ["D", "E", "F"],
}
VARIANTS = [("A", "D", "G"), ("B", "E", "F"),
            ("C", "F", "E"), ("E", "G", "D")]

def make_variant(student_id):
    if not str(student_id).strip():
        raise ValueError("Enter your student ID first")
    variant = sum(ord(ch) for ch in str(student_id).strip()) % 4
    graph = deepcopy(REFERENCE_GRAPH)
    u, v, goal = VARIANTS[variant]
    graph[u].remove(v)
    graph[v].remove(u)
    return {"variant": variant, "graph": graph, "start": "S",
            "goal": goal, "removed_edge": (u, v)}

STUDENT_ID = "09191049"  # TODO: your student ID as text, preserving leading zeros.
if STUDENT_ID.strip():
    assigned = make_variant(STUDENT_ID)
    print(assigned)
else:
    print("Unfinished template: enter your ID to generate your variant.")
print("Python:", platform.python_version())


{'variant': 1, 'graph': {'S': ['A', 'B'], 'A': ['S', 'C', 'D'], 'B': ['S'], 'C': ['A', 'F'], 'D': ['A', 'G'], 'E': ['G'], 'F': ['C', 'G'], 'G': ['D', 'E', 'F']}, 'start': 'S', 'goal': 'F', 'removed_edge': ('B', 'E')}
Python: 3.13.18


### Supplied heuristic data for Task 6

These are shortest distances in the reference graph with all original roads open. Your assigned variant removes a road, so those relaxed-world distances cannot overestimate the remaining route cost. They are supplied task data; students implement the actual BFS and A* algorithms. For goal F: S=3, A=2, B=3, C=1, D=2, E=2, F=0, G=1. No extra hand-trace task is required.

In [36]:
# Supplied heuristic data: h is distance in the original graph before a road closure.
# Removing roads cannot make a route shorter, so these values remain lower bounds.
# h(goal) = 0. These tables apply to the original graph and its edge-deletion variants.
# If you add shortcuts/change costs, re-justify h or use h = 0 instead.
HEURISTICS = {'A': {'A': 0, 'B': 2, 'C': 1, 'D': 1, 'E': 3, 'F': 2, 'G': 2, 'S': 1}, 'B': {'A': 2, 'B': 0, 'C': 3, 'D': 3, 'E': 1, 'F': 3, 'G': 2, 'S': 1}, 'C': {'A': 1, 'B': 3, 'C': 0, 'D': 2, 'E': 3, 'F': 1, 'G': 2, 'S': 2}, 'D': {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}, 'E': {'A': 3, 'B': 1, 'C': 3, 'D': 2, 'E': 0, 'F': 2, 'G': 1, 'S': 2}, 'F': {'A': 2, 'B': 3, 'C': 1, 'D': 2, 'E': 2, 'F': 0, 'G': 1, 'S': 3}, 'G': {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 0, 'S': 3}, 'S': {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2, 'F': 3, 'G': 3, 'S': 0}}
print("Heuristic tables available for targets:", ", ".join(sorted(HEURISTICS)))


Heuristic tables available for targets: A, B, C, D, E, F, G, S


In [37]:
import heapq

h_values = HEURISTICS[goal]

def astar(graph, start, goal, h):
    pq = [(h[start], 0, [start])]
    best_g = {start: 0}
    while pq:
        f, g, path = heapq.heappop(pq)
        node = path[-1]
        if node == goal:
            return path, g
        for nxt in graph[node]:
            new_g = g + 1
            if nxt not in best_g or new_g < best_g[nxt]:
                best_g[nxt] = new_g
                heapq.heappush(pq, (new_g + h[nxt], new_g, path + [nxt]))
    return None, None

print(astar(graph, start, goal, h_values))

(['S', 'A', 'C', 'F'], 3)


## 4. Implement and test the simulator

Write functions for legal moves, state updates, service action and task completion. Reject invalid actions clearly. Do not mutate shared map data between tests. Search plans the route; the simulator verifies/exercises its moves and final service.


In [38]:
# Task 4: simulator (state = (location, serviced))

def legal_moves(state, graph):
    location, serviced = state
    return list(graph[location])

def move(state, nxt, graph):
    location, serviced = state
    if nxt not in graph[location]:
        raise ValueError(f"Invalid move: {location} -> {nxt}")
    return (nxt, serviced)          # نرجع حالة جديدة، ما نغير القديمة

def service(state, goal):
    location, serviced = state
    if location != goal:
        raise ValueError(f"Cannot service at {location}, goal is {goal}")
    return (location, True)

def task_complete(state, goal):
    return state == (goal, True)

def run_route(route, graph, start, goal):
    state = (start, False)
    for nxt in route[1:]:
        state = move(state, nxt, graph)
    state = service(state, goal)
    return state, task_complete(state, goal)

# اختبارات
s0 = (start, False)
print("Legal moves from start:", legal_moves(s0, graph))
print("Valid move:", move(s0, legal_moves(s0, graph)[0], graph))

for name, test in [
    ("Invalid move", lambda: move(s0, goal, graph)),
    ("Service too early", lambda: service(s0, goal)),
]:
    try:
        test()
    except ValueError as e:
        print(f"{name} rejected:", e)

Legal moves from start: ['A', 'B']
Valid move: ('A', False)
Invalid move rejected: Invalid move: S -> F
Service too early rejected: Cannot service at S, goal is F


## 5. Uninformed search: BFS

Implement BFS using a FIFO queue, repeated-state tracking and route reconstruction. Explain why it is uninformed and why it finds a minimum-move path on these unit-cost graphs. Show the route returned for your assigned mission and the agent executing its service action. Handle start=goal, no route and invalid locations correctly. No separate test-case table or results-analysis task is required.

In [39]:
from collections import deque

def bfs(graph, start, goal):
    queue = deque([[start]])   # طابور فيه مسارات
    visited = {start}          # النقاط اللي زرناها
    while queue:
        path = queue.popleft() # نأخذ أقدم مسار (FIFO)
        node = path[-1]        # آخر نقطة فيه
        if node == goal:
            return path
        for nxt in graph[node]:
            if nxt not in visited:
                visited.add(nxt)
                queue.append(path + [nxt])
    return None

bfs_path = bfs(graph, start, goal)
print("BFS path:", bfs_path)
print("Steps:", len(bfs_path) - 1)
print("Simulator check:", run_route(bfs_path, graph, start, goal))

BFS path: ['S', 'A', 'C', 'F']
Steps: 3
Simulator check: (('F', True), True)


## BFS Explanation

**How it works:**
BFS يبدأ من S ويستكشف النقاط طبقة طبقة: أول كل جيران S (A وB)، بعدها جيران جيرانهم، وهكذا. يستخدم queue من نوع FIFO (أول داخل أول خارج)، فالمسارات الأقصر تتعالج قبل الأطول.

**Visited:**
أحفظ النقاط اللي زرتها في مجموعة `visited` عشان ما أرجع لنفس النقطة مرتين وما ندخل في حلقة لا نهائية.

**Route reconstruction:**
كل عنصر في الـ queue هو مسار كامل من S لين النقطة الحالية. لما نوصل للهدف F نرجع المسار كما هو.

**Result:**
['S', 'A', 'C', 'F'] بـ 3 خطوات، والـ simulator تأكد إنه قانوني وتم التسليم.

**Why shortest:**
لأن كل طريق تكلفته 1 وBFS يفحص كل المسارات بطول 1، ثم 2، ثم 3، فأول مرة يوصل للهدف يكون بأقل عدد خطوات.

## 6. Informed search: A*

Implement A* with f(n) = g(n) + h(n), using a priority queue, best-cost tracking and parent reconstruction. Use the supplied HEURISTICS table for your assigned goal. Explain what g and h mean, why the supplied h is admissible on the provided variant, and when it would need revising. Test the goal on removal from the frontier. Show the assigned route and execute the service action. A* and BFS should find the same minimum move count here; their paths may differ. No separate comparison table or limitations essay is required.

In [40]:
import heapq

h_values = HEURISTICS[goal]

def astar(graph, start, goal, h):
    pq = [(h[start], 0, [start])]   # (f, g, المسار)
    best_g = {start: 0}
    while pq:
        f, g, path = heapq.heappop(pq)
        node = path[-1]
        if node == goal:            # نفحص الهدف عند الإخراج من الطابور
            return path, g
        for nxt in graph[node]:
            new_g = g + 1
            if nxt not in best_g or new_g < best_g[nxt]:
                best_g[nxt] = new_g
                heapq.heappush(pq, (new_g + h[nxt], new_g, path + [nxt]))
    return None, None

astar_path, cost = astar(graph, start, goal, h_values)
print("A* path:", astar_path)
print("Cost:", cost)
print("Simulator check:", run_route(astar_path, graph, start, goal))

A* path: ['S', 'A', 'C', 'F']
Cost: 3
Simulator check: (('F', True), True)


•	g: كم مشينا فعلياً من S لين هنا.
	•	h: تقدير الاستاذ لبعد النقطة عن الهدف F.
	•	ليش الـ heuristic سليم: محسوب على الخريطة الأصلية، وقفل طريق ما يقصّر المسافة أبداً، فالتقدير ما يبالغ، وما نحتاج نرجع نراجع نقاط سبق فحصناها.

## 10. Sources, assistance and questions

Cite adapted course code and explain your changes. Disclose AI tool/help/verification or state none used. Record any unresolved questions. Prepare to explain BFS, A*, the heuristic and your simulator after the professor returns.

**Sources:** BFS and A* follow the ideas from the course study notebooks (graph-search and informed-search). The supplied map and heuristic data come from the CS453 workbook.

**AI assistance:** I used Claude (AI assistant) to help me understand the assignment and the algorithms, and it gave me draft code for the simulator, BFS and A*. I ran every cell myself, checked the outputs (BFS and A* both gave ['S','A','C','F'] with cost 3, and the simulator rejected the invalid move and early service), and I am studying the code so I can explain it.

**Unresolved questions:** ( مش فاهمه طريقة التاسك الخامس والسادس حاولت نحله صح منعرفش هل صح ام لا لان حليت من ال ai
)

## GitHub + Colab workflow

1. On the public starter, use Use this template > Create a new repository, choose Private, then invite the professor using the announced GitHub username.
2. Your copy already includes project.ipynb. Open it in Colab using authorized GitHub access or download/upload.
3. Save each work session explicitly to GitHub: File > Save a copy in/to GitHub, correct repo/branch and project.ipynb. Drive autosave alone does not update GitHub.
4. If integration fails, download the notebook from Colab, upload it to GitHub to replace project.ipynb, and commit. No terminal or embedded tokens are needed.
5. Commit real progress throughout the project. Complete README.md using the supplied template.
6. Reopen the committed notebook in a fresh Colab session and verify it runs. Submit repo URL + final commit URL in Classroom and Turn in.

See the student guide pages 5-6 and docs/github-colab-guide.md for details. The professor must accept the invitation before viewing private work; do not wait for acceptance to begin.


## 11. Submission checklist

- [x] Identity, chosen project and generated variant recorded
- [ ] Task 1: PEAS and environment explained
- [ ] Task 2: state, legal actions, transition, goal and costs defined
- [x] Task 4: legal movement, service and task goal implemented
- [x] Task 5: working BFS and assigned mission output
- [x] Task 6: working A*, admissible heuristic explanation and assigned mission output
- [ ] Sources and assistance disclosed
- [ ] README with task/variant/files/dependencies/run steps
- [ ] Clean restart/run-all checked and saved
- [ ] project.ipynb and README.md saved in my private GitHub repository
- [ ] Professor invited; repository URL and final commit URL submitted in Classroom

Original task numbers are retained for reference. Tasks 3, 7, 8 and 9 are removed. There is no Day 7 checkpoint.